# LIAR — Faithfulness Evaluation

Evaluates whether Gemini's LLM-generated explanations are actually faithful to the SHAP attributions they're based on, using the three metrics:

1. **Token coverage:** Does the explanation mention the top SHAP tokens?
2. **Directional accuracy:** Does the explanation correctly represent which direction each mentioned token pushes (toward fake vs real)?
3. **Consistency:** Running the same prediction through the explanation pipeline multiple times — do the explanations stay consistent, or vary wildly?

Builds on the working pipeline from `liar_full_llm_pipeline.ipynb` — reuses the same model loading, SHAP setup, and Gemini API connection.

## 1. Load Model, Data, SHAP, and Gemini


In [1]:
import pandas as pd
import numpy as np
import torch
import shap
import time
import re

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

test_df = pd.read_csv("liar_test.csv")
MAX_LENGTH = 64

from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

distilbert_tokenizer = DistilBertTokenizerFast.from_pretrained("./distilbert_liar_final")
distilbert_model = DistilBertForSequenceClassification.from_pretrained("./distilbert_liar_final")
distilbert_model.to(device)
distilbert_model.eval()

def distilbert_predict_proba(texts, batch_size=8):
    all_probs = []
    texts = list(texts)
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = distilbert_tokenizer(batch, truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = distilbert_model(**inputs).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
        all_probs.append(probs)
        del inputs, logits
        torch.cuda.empty_cache()
    return np.vstack(all_probs)

distilbert_probs = distilbert_predict_proba(test_df["text"].astype(str).tolist())
test_df["distilbert_pred"] = distilbert_probs.argmax(axis=1)
test_df["distilbert_confidence"] = distilbert_probs.max(axis=1)

print(f"DistilBERT test accuracy: {(test_df['distilbert_pred'] == test_df['label_id']).mean():.4f}")


Using device: cuda


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBERT test accuracy: 0.6886


In [2]:
masker = shap.maskers.Text(distilbert_tokenizer)
explainer = shap.Explainer(distilbert_predict_proba, masker)

def get_top_shap_tokens(text, class_idx, top_k=5):
    shap_values = explainer([text])
    tokens = shap_values.data[0]
    values = shap_values.values[0, :, class_idx]
    pairs = list(zip(tokens, values))
    pairs.sort(key=lambda x: abs(x[1]), reverse=True)
    return pairs[:top_k]

print("SHAP explainer ready.")


SHAP explainer ready.


In [3]:
from google import genai
import os

API_KEY = os.environ.get("GEMINI_API_KEY", "AQ.Ab8RN6Kkv38FIWJHo9_STGeuijYlLntPXFYffX62JfmukPtlhw")
client = genai.Client(api_key=API_KEY)
GEMINI_MODEL = "gemini-flash-lite-latest"

def build_explanation_prompt(text, predicted_label, confidence, top_tokens):
    fake_words = [t.strip() for t, v in top_tokens if v > 0]
    real_words = [t.strip() for t, v in top_tokens if v < 0]

    prompt = f"""The model classified this statement as [{predicted_label.upper()}] with {confidence:.0%} confidence.

Statement: "{text}"

The most influential words pushing toward FAKE were: {fake_words if fake_words else "none"}.
The most influential words pushing toward REAL were: {real_words if real_words else "none"}.

Write a 2-sentence explanation for a non-technical reader, referencing the specific influential words above. Do not introduce reasoning that isn't grounded in these words."""
    return prompt

def generate_explanation(prompt, retries=3):
    for attempt in range(retries):
        try:
            response = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
            return response.text.strip()
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(5)
            else:
                raise

test_response = client.models.generate_content(model=GEMINI_MODEL, contents="Reply with exactly: API connection successful.")
print(test_response.text)


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


API connection successful.


## 2. Select Evaluation Sample

Uses disagreement cases — 20 cases for a meaningful sample size without excessive API calls.


In [4]:
baseline_needed = "baseline_pred" not in test_df.columns
if baseline_needed:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline
    train_df = pd.read_csv("liar_train.csv")
    baseline_pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1, 2), stop_words="english", min_df=2)),
        ("clf", LogisticRegression(max_iter=1000, random_state=42))
    ])
    baseline_pipeline.fit(train_df["text"], train_df["label_id"])
    test_df["baseline_pred"] = baseline_pipeline.predict(test_df["text"].astype(str))

test_df["disagreement"] = test_df["baseline_pred"] != test_df["distilbert_pred"]
disagreement_df = test_df[test_df["disagreement"]].copy()

SAMPLE_SIZE = 20
eval_sample = disagreement_df.head(SAMPLE_SIZE).reset_index(drop=True)
print(f"Evaluation sample size: {len(eval_sample)}")


Evaluation sample size: 20


## 3. Metric 1 — Token Coverage

For each case: extract top-5 SHAP tokens, generate the explanation, check how many of those tokens (or close matches) appear in the explanation text.


In [5]:
def token_coverage(explanation_text, top_tokens):
    explanation_lower = explanation_text.lower()
    mentioned = 0
    for token, val in top_tokens:
        clean_token = re.sub(r"[^a-zA-Z0-9]", "", token.strip().lower())
        if len(clean_token) > 1 and clean_token in explanation_lower:
            mentioned += 1
    return mentioned / len(top_tokens) if top_tokens else 0.0

coverage_results = []

for idx, row in eval_sample.iterrows():
    predicted_label = "fake" if row["distilbert_pred"] == 1 else "real"
    top_tokens = get_top_shap_tokens(row["text"], class_idx=int(row["distilbert_pred"]), top_k=5)
    prompt = build_explanation_prompt(row["text"], predicted_label, row["distilbert_confidence"], top_tokens)
    explanation = generate_explanation(prompt)

    coverage = token_coverage(explanation, top_tokens)

    coverage_results.append({
        "text": row["text"],
        "predicted_label": predicted_label,
        "top_tokens": top_tokens,
        "explanation": explanation,
        "token_coverage": coverage,
    })

    print(f"[{idx+1}/{len(eval_sample)}] Coverage: {coverage:.2f}")
    time.sleep(4.5)  # stay under 15 requests/minute

coverage_df = pd.DataFrame([
    {"text": r["text"], "predicted_label": r["predicted_label"], "token_coverage": r["token_coverage"]}
    for r in coverage_results
])

print(f"\nMean token coverage: {coverage_df['token_coverage'].mean():.2%}")
coverage_df


[1/20] Coverage: 1.00


[2/20] Coverage: 0.80


[3/20] Coverage: 0.80


[4/20] Coverage: 0.80


[5/20] Coverage: 0.80


...

[18/20] Coverage: 1.00


[19/20] Coverage: 1.00


[20/20] Coverage: 1.00



Mean token coverage: 78.00%


,text,predicted_label,token_coverage
0,Wisconsin is on pace to double the number of l...,fake,1.0
1,There have not been any public safety issues i...,fake,0.8
2,The number of illegal immigrants could be 3 mi...,fake,0.8
3,"Now, there was a time when someone like Scalia...",real,0.8
4,Its been since 1888 that a Senate of a differe...,real,0.8
5,"Under Rosemary Lehmberg, the Travis County D.A...",fake,0.4
6,Says he won the second debate with Hillary Cli...,real,0.8
7,ACORN will be a paid partner with the Census B...,fake,1.0
8,Says Marco Rubio said Social Security and Medi...,real,1.0
9,What the facts say is ...the best scenario for...,fake,0.6


## 4. Metric 2 — Directional Accuracy

For each mentioned token, check whether the explanation correctly represents its direction (toward fake vs toward real), by checking simple sentence-level context around the mention.


In [6]:
def directional_accuracy(explanation_text, top_tokens):
    """
    Simple heuristic: for each mentioned token, check whether it appears closer to
    'fake'-associated or 'real'-associated language in the explanation, and compare
    against its actual SHAP direction.
    """
    explanation_lower = explanation_text.lower()
    sentences = re.split(r'(?<=[.!?])\s+', explanation_text)

    correct = 0
    total_checked = 0

    for token, val in top_tokens:
        clean_token = re.sub(r"[^a-zA-Z0-9]", "", token.strip().lower())
        if len(clean_token) <= 1 or clean_token not in explanation_lower:
            continue

        actual_direction = "fake" if val > 0 else "real"

        # Find the sentence(s) mentioning this token
        relevant_sentences = [s for s in sentences if clean_token in s.lower()]
        if not relevant_sentences:
            continue

        context = " ".join(relevant_sentences).lower()
        mentions_fake = "fake" in context or "false" in context
        mentions_real = "real" in context or "true" in context

        # Determine what direction the explanation associates this token with
        if mentions_fake and not mentions_real:
            stated_direction = "fake"
        elif mentions_real and not mentions_fake:
            stated_direction = "real"
        else:
            continue  # ambiguous, skip

        total_checked += 1
        if stated_direction == actual_direction:
            correct += 1

    return correct / total_checked if total_checked > 0 else None

directional_results = []
for r in coverage_results:
    acc = directional_accuracy(r["explanation"], r["top_tokens"])
    directional_results.append(acc)

valid_scores = [s for s in directional_results if s is not None]
print(f"Directional accuracy computed on {len(valid_scores)}/{len(directional_results)} cases (others had ambiguous/unclear direction language)")
print(f"Mean directional accuracy: {np.mean(valid_scores):.2%}" if valid_scores else "No valid cases to compute.")


Directional accuracy computed on 15/20 cases (others had ambiguous/unclear direction language)
Mean directional accuracy: 93.33%


## 5. Metric 3 — Consistency

Run the same prediction through the explanation pipeline 5 times, and measure how much the generated explanations vary. Uses word-overlap similarity across the 5 runs as a simple consistency proxy.


In [7]:
def run_consistency_test(row, n_runs=5):
    predicted_label = "fake" if row["distilbert_pred"] == 1 else "real"
    top_tokens = get_top_shap_tokens(row["text"], class_idx=int(row["distilbert_pred"]), top_k=5)
    prompt = build_explanation_prompt(row["text"], predicted_label, row["distilbert_confidence"], top_tokens)

    explanations = []
    for _ in range(n_runs):
        exp = generate_explanation(prompt)
        explanations.append(exp)
        time.sleep(4.5)

    return explanations

def explanation_similarity(explanations):
    """Average pairwise word-overlap (Jaccard similarity) across all explanation pairs."""
    word_sets = [set(re.findall(r"[a-zA-Z']+", exp.lower())) for exp in explanations]
    similarities = []
    for i in range(len(word_sets)):
        for j in range(i + 1, len(word_sets)):
            intersection = len(word_sets[i] & word_sets[j])
            union = len(word_sets[i] | word_sets[j])
            similarities.append(intersection / union if union > 0 else 0)
    return np.mean(similarities) if similarities else 0.0

# Run on a small number of cases — this uses n_runs API calls per case
CONSISTENCY_SAMPLE_SIZE = 3
consistency_results = []

for idx in range(min(CONSISTENCY_SAMPLE_SIZE, len(eval_sample))):
    row = eval_sample.iloc[idx]
    explanations = run_consistency_test(row, n_runs=5)
    similarity = explanation_similarity(explanations)

    consistency_results.append({
        "text": row["text"],
        "explanations": explanations,
        "avg_similarity": similarity,
    })

    print(f"Case {idx+1}: avg pairwise similarity = {similarity:.2%}")

overall_consistency = np.mean([r["avg_similarity"] for r in consistency_results])
print(f"\nOverall consistency score: {overall_consistency:.2%}")


Case 1: avg pairwise similarity = 48.08%


Case 2: avg pairwise similarity = 32.72%


Case 3: avg pairwise similarity = 37.04%

Overall consistency score: 39.28%


## 6. Faithfulness Summary


In [8]:
faithfulness_summary = {
    "token_coverage_mean": coverage_df["token_coverage"].mean(),
    "directional_accuracy_mean": np.mean(valid_scores) if valid_scores else None,
    "consistency_mean": overall_consistency,
    "n_coverage_cases": len(coverage_df),
    "n_directional_cases": len(valid_scores),
    "n_consistency_cases": len(consistency_results),
}

import json
with open("liar_faithfulness_summary.json", "w") as f:
    json.dump(faithfulness_summary, f, indent=2, default=float)

coverage_df.to_csv("liar_faithfulness_token_coverage.csv", index=False)

print("Faithfulness Evaluation Summary:")
for k, v in faithfulness_summary.items():
    print(f"  {k}: {v}")


Faithfulness Evaluation Summary:
  token_coverage_mean: 0.78
  directional_accuracy_mean: 0.9333333333333333
  consistency_mean: 0.3927678837127932
  n_coverage_cases: 20
  n_directional_cases: 15
  n_consistency_cases: 3


## 7. Semantic Consistency (Sentence Embeddings)

Word overlap treats paraphrasing as inconsistency, which isn't a fair measure. This replaces it with semantic similarity using sentence embeddings (`all-MiniLM-L6-v2`) — measuring whether the explanations convey the same *meaning* across the 5 runs, even if worded differently.

Requires: `pip install sentence-transformers`


In [9]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

embedder = SentenceTransformer("all-MiniLM-L6-v2")

def semantic_consistency(explanations):
    """Average pairwise cosine similarity across all explanation pairs, using sentence embeddings."""
    embeddings = embedder.encode(explanations)
    sim_matrix = cosine_similarity(embeddings)

    similarities = []
    n = len(explanations)
    for i in range(n):
        for j in range(i + 1, n):
            similarities.append(sim_matrix[i][j])

    return np.mean(similarities) if similarities else 0.0


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
semantic_scores = []

for r in consistency_results:
    sem_score = semantic_consistency(r["explanations"])
    semantic_scores.append(sem_score)
    print(f"Case: {r['text'][:60]}...")
    print(f"  Word-overlap consistency: {r['avg_similarity']:.2%}")
    print(f"  Semantic consistency (cosine sim): {sem_score:.3f}")
    print()

overall_semantic_consistency = np.mean(semantic_scores)
print(f"Overall semantic consistency: {overall_semantic_consistency:.3f}")


Case: Wisconsin is on pace to double the number of layoffs this ye...
  Word-overlap consistency: 48.08%
  Semantic consistency (cosine sim): 0.898

Case: There have not been any public safety issues in cities that ...
  Word-overlap consistency: 32.72%
  Semantic consistency (cosine sim): 0.841

Case: The number of illegal immigrants could be 3 million. It coul...
  Word-overlap consistency: 37.04%
  Semantic consistency (cosine sim): 0.837

Overall semantic consistency: 0.859


### 7.1 Example: Same Meaning, Different Words

In [11]:
example_case = consistency_results[0]
print("Statement:", example_case["text"])
print(f"Semantic consistency for this case: {semantic_scores[0]:.3f}")
print(f"Word-overlap consistency for this case: {example_case['avg_similarity']:.2%}")
print()
for i, exp in enumerate(example_case["explanations"], 1):
    print(f"Run {i}: {exp}")
    print()


Statement: Wisconsin is on pace to double the number of layoffs this year.
Semantic consistency for this case: 0.898
Word-overlap consistency for this case: 48.08%

Run 1: The model leaned toward classifying the statement as fake because of the words "Wisconsin" and "lay." Conversely, the words "pace," "double," and "year" pushed the model slightly toward considering the statement real.

Run 2: The model leaned toward classifying the statement as fake because of the specific mention of geographic and economic terms like **"Wisconsin"** and **"lay"** (referring to layoffs). Conversely, the words **"pace"**, **"double"**, and **"year"** pushed the model slightly toward believing the statement was real.

Run 3: The model leaned toward classifying the statement as [FAKE] largely because of the specific locations and topics mentioned by the words "**Wisconsin**" and "**lay**." Conversely, terms like "**pace**," "**double**," and "**year**" pushed the model slightly toward considering it [RE

### 7.2 Updated Faithfulness Summary

In [ ]:
faithfulness_summary["semantic_consistency_mean"] = float(overall_semantic_consistency)
faithfulness_summary["word_overlap_consistency_mean"] = float(overall_consistency)

with open("liar_faithfulness_summary.json", "w") as f:
    json.dump(faithfulness_summary, f, indent=2, default=float)

print("Final Faithfulness Evaluation Summary:")
for k, v in faithfulness_summary.items():
    print(f"  {k}: {v}")

print()
if overall_semantic_consistency > 0.8:
    print(f'Framing: "Explanations are semantically stable but lexically diverse — cosine similarity of '
          f'{overall_semantic_consistency:.2f} despite only {overall_consistency:.0%} word overlap, indicating '
          f'consistent meaning conveyed through natural paraphrasing rather than unfaithful or random variation."')
else:
    print(f"Semantic consistency ({overall_semantic_consistency:.2f}) is below the 0.8 threshold "
          f"as 'high' — worth reviewing example cases manually before finalizing this framing.")


Final Faithfulness Evaluation Summary:
  token_coverage_mean: 0.78
  directional_accuracy_mean: 0.9333333333333333
  consistency_mean: 0.3927678837127932
  n_coverage_cases: 20
  n_directional_cases: 15
  n_consistency_cases: 3
  semantic_consistency_mean: 0.8586254119873047
  word_overlap_consistency_mean: 0.3927678837127932

Framing: "Explanations are semantically stable but lexically diverse — cosine similarity of 0.86 despite only 39% word overlap, indicating consistent meaning conveyed through natural paraphrasing rather than unfaithful or random variation."
